# A2A Agent Card 与 Task 生命周期：差旅委派事件实战

**面试问题：Agent 怎样发现彼此、协商能力，并管理 task、message、artifact、取消和终态？**

## 回答主线

先把真实请求和资源合同摆出来，用最简单的方案建立成本或正确性基线，再手写核心控制逻辑并展示完整事件、指标和失败修正。断言只在最后保护少量关键不变量，前面的可见输入、过程和结果才是学习主体。

## 真实案例

企业差旅主管 Agent 要把“寻找上海两晚、预算 1800 元、可取消酒店”委派给酒店 Agent。案例使用可读 Agent Card、消息和报价 artifact，对比自由文本一问一答与显式 task 状态机，展示能力协商、流式事件、artifact 校验、取消幂等以及不允许从终态重新运行的失败路径。

### 输入预览：用户意图与两个 Agent Card

In [1]:
from pprint import pprint  # 导入结构化打印工具展示协议对象。

user_request = {"city": "上海", "nights": 2, "budget_cny": 1800, "refundable": True, "check_in": "2026-08-12"}  # 构造具有真实约束的差旅请求。
agent_cards = {  # 定义主管和酒店两个 Agent 的发现元数据。
    "travel-supervisor": {"version": "1.0", "skills": ["plan_trip", "approve_budget"], "streaming": True, "auth": "oauth2"},  # 主管负责预算和最终计划。
    "hotel-search": {"version": "1.1", "skills": ["search_hotel", "cancel_task"], "streaming": True, "auth": "oauth2"},  # 酒店 Agent 支持搜索与取消。
}  # 完成 Agent Card registry。
print("用户差旅请求：")  # 输出真实任务输入标题。
pprint(user_request, sort_dicts=False)  # 展示城市、日期、预算和退款约束。
print("可发现 Agent Card：")  # 输出能力发现标题。
pprint(agent_cards, sort_dicts=False)  # 展示版本、技能、流式和认证能力。

用户差旅请求：
{'city': '上海',
 'nights': 2,
 'budget_cny': 1800,
 'refundable': True,
 'check_in': '2026-08-12'}
可发现 Agent Card：
{'travel-supervisor': {'version': '1.0',
                       'skills': ['plan_trip', 'approve_budget'],
                       'streaming': True,
                       'auth': 'oauth2'},
 'hotel-search': {'version': '1.1',
                  'skills': ['search_hotel', 'cancel_task'],
                  'streaming': True,
                  'auth': 'oauth2'}}


## Baseline 基线：自由文本调用只返回一个字符串

In [2]:
def free_form_delegate(request):  # 构造没有任务标识、状态和 artifact 的脆弱基线。
    return f'找到上海酒店，约 {request["budget_cny"]} 元以内，请稍候'  # 返回无法验证来源、约束和完成状态的自由文本。

baseline_reply = free_form_delegate(user_request)  # 执行一问一答式委派。
print("自由文本响应：", baseline_reply)  # 展示看似自然但没有机器可验证语义的结果。
print("基线缺失：task_id、状态、消息角色、artifact schema、取消句柄和终态原因。")  # 明确为什么字符串不能支撑长时任务。

自由文本响应： 找到上海酒店，约 1800 元以内，请稍候
基线缺失：task_id、状态、消息角色、artifact schema、取消句柄和终态原因。


### 核心实现：显式 Task、Message 与事件状态机

In [3]:
ALLOWED = {"submitted": {"working", "canceled"}, "working": {"input-required", "completed", "failed", "canceled"}, "input-required": {"working", "canceled"}, "completed": set(), "failed": set(), "canceled": set()}  # 定义任务状态的合法有向转换。

def new_task(task_id, sender, receiver, request):  # 创建带身份、输入和事件账本的 A2A 任务。
    return {"id": task_id, "sender": sender, "receiver": receiver, "state": "submitted", "request": request.copy(), "messages": [], "artifacts": [], "events": [(0, "submitted", "任务已登记")]}  # 返回完整可持久化任务对象。

def transition(task, new_state, detail):  # 对任务执行带规则检查的状态转换。
    if new_state not in ALLOWED[task["state"]]:  # 阻止终态复活和未定义跳转。
        raise ValueError(f'非法状态转换 {task["state"]}->{new_state}')  # 返回可诊断协议错误。
    task["state"] = new_state  # 更新当前权威状态。
    task["events"].append((len(task["events"]), new_state, detail))  # 追加而非覆盖事件历史。

task = new_task("task-hotel-7842", "travel-supervisor", "hotel-search", user_request)  # 创建差旅委派任务。
transition(task, "working", "酒店 Agent 已接受任务并开始检索")  # 记录远端 Agent 接受任务。
task["messages"].append({"role": "agent", "text": "正在比较可取消房型", "sequence": 1})  # 保存第一条流式进度消息。
task["messages"].append({"role": "agent", "text": "找到 3 个预算内候选", "sequence": 2})  # 保存第二条带顺序号的进度消息。
print("任务事件和流式消息：")  # 输出长时任务的可观察过程。
for event in task["events"]:  # 逐项展示状态历史。
    print("event", event)  # 显示 submitted 到 working 的转换。
for message in task["messages"]:  # 逐项展示有序流式消息。
    print("message", message)  # 显示客户端可恢复的 sequence。

任务事件和流式消息：
event (0, 'submitted', '任务已登记')
event (1, 'working', '酒店 Agent 已接受任务并开始检索')
message {'role': 'agent', 'text': '正在比较可取消房型', 'sequence': 1}
message {'role': 'agent', 'text': '找到 3 个预算内候选', 'sequence': 2}


### Artifact 验收与任务完成

In [4]:
offers = [  # 构造酒店 Agent 返回的结构化报价 artifact。
    {"hotel": "外滩商务酒店", "total_cny": 1680, "refundable": True, "source": "supplier-A", "quote_id": "q-101"},  # 满足预算和可取消约束的候选。
    {"hotel": "静安精品酒店", "total_cny": 1760, "refundable": True, "source": "supplier-B", "quote_id": "q-102"},  # 第二个满足条件的候选。
    {"hotel": "浦东景观酒店", "total_cny": 1520, "refundable": False, "source": "supplier-C", "quote_id": "q-103"},  # 价格低但违反可取消条件的反例。
]  # 完成候选报价列表。
valid_offers = [offer for offer in offers if offer["total_cny"] <= task["request"]["budget_cny"] and (not task["request"]["refundable"] or offer["refundable"])]  # 用原始用户约束过滤 artifact。
artifact = {"type": "hotel-offers/v1", "task_id": task["id"], "offers": valid_offers, "generated_by": task["receiver"]}  # 构造绑定任务和生成方的 artifact。
task["artifacts"].append(artifact)  # 把结构化结果附加到任务而不是塞入自由文本。
transition(task, "completed", f"返回 {len(valid_offers)} 个通过约束的报价")  # 只有 artifact 验收后才能进入完成终态。
print("完成 artifact：")  # 输出可由上游程序继续消费的结果。
pprint(artifact, sort_dicts=False)  # 展示类型、任务绑定、候选和来源。
print("最终状态：", task["state"], "终态事件：", task["events"][-1])  # 展示功能完成与状态完成保持一致。

完成 artifact：
{'type': 'hotel-offers/v1',
 'task_id': 'task-hotel-7842',
 'offers': [{'hotel': '外滩商务酒店',
             'total_cny': 1680,
             'refundable': True,
             'source': 'supplier-A',
             'quote_id': 'q-101'},
            {'hotel': '静安精品酒店',
             'total_cny': 1760,
             'refundable': True,
             'source': 'supplier-B',
             'quote_id': 'q-102'}],
 'generated_by': 'hotel-search'}
最终状态： completed 终态事件： (2, 'completed', '返回 2 个通过约束的报价')


## 结果解读：协议对象解决了哪些真实问题

In [5]:
comparison = [  # 构造自由文本基线与 A2A 任务的能力对照。
    ("恢复进度", False, True),  # sequence 和事件账本允许断线后恢复。
    ("校验预算/退款", False, True),  # 结构化 artifact 可以执行确定性约束。
    ("安全取消", False, True),  # task_id 和状态机支持幂等取消。
    ("审计生成方", False, True),  # sender、receiver 和 generated_by 保存来源。
]  # 完成对照表数据。
print("能力             自由文本  A2A任务")  # 输出方案对照表表头。
for capability, baseline_ok, task_ok in comparison:  # 逐项展示显式协议带来的能力。
    print(f"{capability:<16} {str(baseline_ok):<8} {str(task_ok):<8}")  # 让学习者看到状态机不是形式主义。
print("解读：Agent Card 只负责发现与能力声明；每次任务仍需真实认证、授权、输入验证和 artifact 验收。")  # 区分发现协议和执行授权。

能力             自由文本  A2A任务
恢复进度             False    True    
校验预算/退款          False    True    
安全取消             False    True    
审计生成方            False    True    
解读：Agent Card 只负责发现与能力声明；每次任务仍需真实认证、授权、输入验证和 artifact 验收。


## 失败案例：取消幂等与终态不可复活

In [6]:
cancel_task = new_task("task-hotel-cancel", "travel-supervisor", "hotel-search", user_request)  # 创建一条用于演示取消的独立任务。
transition(cancel_task, "working", "开始搜索")  # 让任务进入运行状态。
transition(cancel_task, "canceled", "用户修改行程")  # 第一次取消进入合法终态。
duplicate_cancel = "already-canceled" if cancel_task["state"] == "canceled" else "unexpected"  # 把重复取消处理为幂等读取而非再次产生副作用。
revive_error = None  # 初始化终态复活错误结果。
try:  # 尝试错误地把已取消任务重新标为运行。
    transition(cancel_task, "working", "错误重试")  # 发起状态机禁止的终态复活。
except ValueError as error:  # 捕获预期协议错误。
    revive_error = str(error)  # 保存错误供调用方诊断和审计。
print("重复取消结果：", duplicate_cancel)  # 展示安全重试不会重复执行取消副作用。
print("终态复活结果：", revive_error)  # 展示非法转换被明确拒绝。

重复取消结果： already-canceled
终态复活结果： 非法状态转换 canceled->working


### 生产边界

In [7]:
protocol_trace = {"task_id": task["id"], "card_version": agent_cards["hotel-search"]["version"], "event_count": len(task["events"]), "message_sequences": [message["sequence"] for message in task["messages"]], "artifact_type": artifact["type"], "terminal": task["state"]}  # 汇总可用于跨 Agent 调试的低敏 trace。
print("协议 trace：", protocol_trace)  # 展示版本、消息顺序、artifact 和终态。
print("生产替换点：还需 A2A wire schema、OAuth audience、流式重连、artifact 存储、任务 TTL、回调签名和跨组织策略。")  # 明确内存字典与真实跨网络协议的差距。

协议 trace： {'task_id': 'task-hotel-7842', 'card_version': '1.1', 'event_count': 3, 'message_sequences': [1, 2], 'artifact_type': 'hotel-offers/v1', 'terminal': 'completed'}
生产替换点：还需 A2A wire schema、OAuth audience、流式重连、artifact 存储、任务 TTL、回调签名和跨组织策略。


## 回归测试：只保护能力、约束和终态

In [8]:
assert "search_hotel" in agent_cards["hotel-search"]["skills"]  # 验证远端 Agent Card 宣告了任务所需能力。
assert task["state"] == "completed"  # 验证只有 artifact 通过约束后任务才进入完成终态。
assert all(offer["refundable"] and offer["total_cny"] <= 1800 for offer in artifact["offers"])  # 验证返回报价满足用户预算与退款约束。
assert [message["sequence"] for message in task["messages"]] == [1, 2]  # 验证流式消息序列连续且可恢复。
assert duplicate_cancel == "already-canceled" and revive_error is not None  # 验证取消幂等且终态不能复活。
print("回归测试通过：能力发现、artifact 约束、消息顺序、完成终态和取消语义均成立。")  # 用少量测试保护协议核心。

回归测试通过：能力发现、artifact 约束、消息顺序、完成终态和取消语义均成立。
